In [27]:
!pip install playwright

In [28]:
!playwright install

Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Missing libraries:                                   ║
║     libwoff2dec.so.1.0.2                             ║
║     libgstgl-1.0.so.0                                ║
║     libgstcodecparsers-1.0.so.0                      ║
║     libavif.so.13                                    ║
║     libharfbuzz-icu.so.0                             ║
║     libenchant-2.so.2                                ║
║     libsecret-1.so.0                                 ║
║     libhyphen.so.0                                   ║
║     libmanette-0.2.so.0                              ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.11/dist-packages/playwright/driver/package/lib/server/registry/dependencies.js:216:9)
    at process.processTicksAndRejections (node:internal/process/task_queues:105

In [29]:
from playwright.async_api import async_playwright
import asyncio
import pandas as pd

URL = "https://solocoffee.su/svezheobzharennyj-kofe-zernovoj"

async def get_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(URL)
        await page.wait_for_selector("div.card")
        cards = await page.query_selector_all("div.card")
        coffee_data = []
        for card in cards:
            name = await card.query_selector("a.card__name")
            name = await name.inner_text()
            link = await card.query_selector("a.card__name")
            link = await link.get_attribute("href")
            compound = await card.query_selector("div.card__compound")
            compound = await compound.inner_text() if compound else "N/A" #вылезла ошибка, что не вездe есть описание((
            description = await card.query_selector("div.card__desc")
            description = await description.inner_text()
            price = await card.query_selector("div.card__price")
            price = await price.inner_text()
            coffee_data.append({
                "Название": name,
                "Ссылка": link,
                "Состав": compound,
                "Описание": description,
                "Цена": price
            })
        df = pd.DataFrame(coffee_data)
        df.to_csv("solo_coffee_data.csv", index=False, encoding="utf-8")
        await browser.close()

await get_data()

In [31]:
database = pd.read_csv("solo_coffee_data.csv")
database["Цена"] = database["Цена"].str.replace(" ₽", " ", regex=False)
database["Цена"] = pd.to_numeric(database["Цена"], errors="coerce")
database.at[25, "Цена"] = database.at[25, "Цена"] / 4
average_price = database["Цена"].mean()
database["Цена за грамм"] = database["Цена"]/250
display(database)
print("Средняя цена за 250 г обжаренного кофе:", average_price)
print("Средневзвешенная цена грамма кофе Solo Coffee:", average_price/250)

<ipython-input-31-30ac68de40cf>:4: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1649.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



,Название,Ссылка,Состав,Описание,Цена,Цена за грамм
0,Blend 001,https://solocoffee.su/svezheobzharennyj-kofe-z...,50% арабика / 50% робуста,"Blend 001 - это сбалансированная смесь, нашей ...",630.00,2.520
1,Blend 002,https://solocoffee.su/svezheobzharennyj-kofe-z...,20% арабика / 80% робуста,"Во вкусе нотки какао, горького шоколада, лесны...",600.00,2.400
2,Blend ARABICA,https://solocoffee.su/svezheobzharennyj-kofe-z...,100% арабика,"Чистая, сладкая и сбалансированная чашка с нот...",670.00,2.680
3,Blend CREMA,https://solocoffee.su/svezheobzharennyj-kofe-z...,20% арабика / 80% робуста,"Во вкусе нотки какао, горького шоколада, лесны...",600.00,2.400
4,Blend GOLD,https://solocoffee.su/svezheobzharennyj-kofe-z...,50% арабика /50% робуста,"Во вкусе нотки горького шоколада, грецкого оре...",630.00,2.520
5,Blend NERO,https://solocoffee.su/svezheobzharennyj-kofe-z...,арабика / робуста (коммерческая тайна),"Кофе с выраженной горчинкой, с землистыми нота...",580.00,2.320
6,Blend PLATINUM,https://solocoffee.su/svezheobzharennyj-kofe-z...,40% арабика / 60% робуста,"Во вкусе нотки какао, молочного шоколада, лесн...",640.00,2.560
7,Crema Gold,https://solocoffee.su/svezheobzharennyj-kofe-z...,арабика / робуста коммерческая тайна,"Crema Gold - Бленд с очень плотным телом, обво...",580.00,2.320
8,Ароматизированный кофе Амаретто,https://solocoffee.su/svezheobzharennyj-kofe-z...,100% арабика,Ароматный кофе со вкусом темного шоколада и ит...,690.00,2.760
9,Ароматизированный кофе Баварский Шоколад,https://solocoffee.su/svezheobzharennyj-kofe-z...,100% арабика,Насыщенный благородный вкус и аромат темного ш...,690.00,2.760


Средняя цена за 250 г обжаренного кофе: 688.3716216216217
Средневзвешенная цена грамма кофе Solo Coffee: 2.7534864864864868


In [32]:
import pandas as pd
import plotly.express as px

fig = px.bar(
    database,
    x="Название",
    y="Цена",
    text="Цена",
    title="Уровень цен на кофе поставщика Solo Coffee (за 250 г)",
    labels={"Цена": "Цена (₽)", "Название": "Название кофе"},
    color="Цена",
    color_continuous_scale="Blues",

)

fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Название кофе",
    yaxis_title="Цена (₽)",
    template="plotly_white",
    width=1000,
    height=1000,
    xaxis_tickangle=-45
)

fig.show()

In [34]:
fig = px.bar(
    database,
    x="Название",
    y="Цена за грамм",
    text="Цена за грамм",
    title="Средневзвешенные цена видов кофе поставщика Solo Coffee",
    labels={"Средневзвешенная цена": "Цена за грамм", "Название": "Название кофе"},
    color="Цена за грамм",
    color_continuous_scale="greens",

)

fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Название кофе",
    yaxis_title="Цена (₽)",
    template="plotly_white",
    width=1000,
    height=1000,
    xaxis_tickangle=-45
)

fig.show()